# bronze2silver_transform_nbk
Python-only notebook (no PySpark). Reads a bronze-partition file in its **native format**, stages it as parquet in the Lakehouse, then loads it into a Fabric **Data Warehouse** silver table using **SCD Type 2** semantics (close-and-insert merge).

**Layer topology**
- Lakehouse = raw bronze files only (parquet from DB sources, original format for file sources)
- Warehouse = all curated layers (silver, gold, platinum) + config tables

**SCD2 columns appended to every silver row**: `row_hash`, `valid_from`, `valid_to`, `is_current`.

In [ ]:
# Cell 1 — Parameters (Fabric injects values at runtime)
bronze_file_type      = ""   # parquet | csv | json | xml | excel
datasubject           = ""
classification        = ""
sourcesystem          = ""
sourceschema          = ""
tablename             = ""
ingest_channel        = ""
ingest_partition      = ""   # e.g. '2026-04-17-14'
container             = ""   # 'bronze' (kept for path compatibility)
criteria_columns      = ""   # CSV list of business-key columns, e.g. 'id,tenant_id'
full_refresh_flag     = "N"  # 'Y' => truncate silver before load

table_id              = ""   # from elt_table_config — used to look up elt_schema_config overrides
file_pattern          = ""   # passed through from config (not used in transform logic)

pipeline_id           = ""
pipeline_name         = ""
pipeline_run_id       = ""
pipeline_trigger_id   = ""
pipeline_trigger_time = ""
pipeline_trigger_type = ""

# Warehouse target
warehouse_endpoint    = "nv3l45p7irxunojdrwsud5otge-isco3hrff5zuzpeqnjwvvo6u5i.datawarehouse.fabric.microsoft.com"
warehouse_name        = "MDF_WAREHOUSE_DEV_001"
silver_schema         = "silver"

# Lakehouse ABFSS (required for Warehouse COPY INTO)
onelake_workspace_id  = "9eed8444-2f25-4c73-bc90-6a6d5abbd4ea"
onelake_lakehouse_id  = "e638c447-1201-4ad5-b5c0-b51f406ec2ec"

In [ ]:
# Cell 2 — Imports
import os, hashlib, struct, uuid, json
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.dataset as pads
import pyodbc
from azure.identity import DefaultAzureCredential


def _connect_warehouse() -> pyodbc.Connection:
    token = DefaultAzureCredential().get_token('https://database.windows.net/.default').token
    token_bytes = bytes(token, 'utf-8')
    exp_token = b''
    for b in token_bytes:
        exp_token += bytes({b}) + bytes(1)
    attrs = {1256: struct.pack('=i', len(exp_token)) + exp_token}  # SQL_COPT_SS_ACCESS_TOKEN
    conn_str = (
        'Driver={ODBC Driver 18 for SQL Server};'
        f'Server={warehouse_endpoint},1433;'
        f'Database={warehouse_name};'
        'Encrypt=yes;TrustServerCertificate=no;Connection Timeout=30;'
    )
    return pyodbc.connect(conn_str, attrs_before=attrs, autocommit=False)


def execute(conn, sql: str, params: tuple = ()):
    with conn.cursor() as cur:
        cur.execute(sql, params) if params else cur.execute(sql)

In [ ]:
# Cell 3 — Derived identifiers and paths
silver_table        = tablename                                       # one-to-one naming
silver_fqn          = f"[{silver_schema}].[{silver_table}]"
staging_table       = f"{silver_table}__stg_{pipeline_run_id.replace('-', '')[:16]}"
staging_fqn         = f"[{silver_schema}].[{staging_table}]"

# Bronze partition folder (Lakehouse local mount)
bronze_rel          = f"Files/bronze/{datasubject}/{classification}/{sourcesystem}/{sourceschema}/{tablename}/{ingest_channel}/{ingest_partition}"
bronze_local        = f"/lakehouse/default/{bronze_rel}"

# Staging parquet (single compacted file per run, consumed by COPY INTO)
staging_rel         = f"Files/_staging/bronze2silver/{tablename}/{pipeline_run_id}.parquet"
staging_local       = f"/lakehouse/default/{staging_rel}"
staging_abfss       = (
    f"abfss://{onelake_workspace_id}@onelake.dfs.fabric.microsoft.com/"
    f"{onelake_lakehouse_id}/{staging_rel}"
)

business_keys = [c.strip() for c in criteria_columns.split(',') if c.strip()]
is_full_refresh = full_refresh_flag.upper() == 'Y'
scd2_enabled    = bool(business_keys) and not is_full_refresh

print(f"bronze_local    = {bronze_local}")
print(f"silver_fqn      = {silver_fqn}")
print(f"business_keys   = {business_keys}")
print(f"scd2_enabled    = {scd2_enabled}")
print(f"full_refresh    = {is_full_refresh}")

In [ ]:
# Cell 4 — Read bronze file(s) into a pandas DataFrame (format-aware)
def read_bronze(path: str, fmt: str) -> pd.DataFrame:
    fmt = (fmt or 'parquet').lower()
    p = Path(path)
    files = sorted([f for f in p.rglob('*') if f.is_file()]) if p.is_dir() else [p]
    if not files:
        raise FileNotFoundError(f"No bronze files under {path}")

    if fmt == 'parquet':
        return pads.dataset([str(f) for f in files], format='parquet').to_table().to_pandas()
    if fmt == 'csv':
        return pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
    if fmt == 'json':
        return pd.concat([pd.read_json(f, lines=True) for f in files], ignore_index=True)
    if fmt == 'xml':
        return pd.concat([pd.read_xml(f) for f in files], ignore_index=True)
    if fmt == 'excel':
        return pd.concat([pd.read_excel(f) for f in files], ignore_index=True)
    raise ValueError(f"Unsupported bronze_file_type '{fmt}'")

df = read_bronze(bronze_local, bronze_file_type)
print(f"bronze rows: {len(df):,}   columns: {len(df.columns)}")

In [ ]:
# Cell 4b — Apply elt_schema_config overrides (sparse: renames + explicit type casts only)
# Rows absent from elt_schema_config are left as-is; DDL builder handles type inference.
_TSQL_TO_PANDAS = {
    'bigint': 'Int64',   'int': 'Int32',   'smallint': 'Int16',  'tinyint': 'UInt8',
    'float': 'float64',  'real': 'float32', 'bit': 'boolean',
    'datetime2': 'datetime64[ns]', 'date': 'datetime64[ns]',
    'varchar': 'object', 'nvarchar': 'object', 'char': 'object',
    'decimal': 'float64', 'numeric': 'float64', 'uniqueidentifier': 'object',
}

if table_id:
    _schema_conn = _connect_warehouse()
    try:
        with _schema_conn.cursor() as _cur:
            _cur.execute(
                "SELECT source_column_name, target_column_name, target_data_type "
                "FROM mdf_platform_orchestration.elt_schema_config "
                "WHERE table_id = ?",
                (int(table_id),)
            )
            _overrides = _cur.fetchall()
    finally:
        _schema_conn.close()

    # apply renames where source and target names differ
    rename_map = {r[0]: r[1] for r in _overrides if r[1] and r[0] != r[1]}
    if rename_map:
        df = df.rename(columns=rename_map)

    # apply explicit type casts
    for src_col, tgt_col, tgt_type in _overrides:
        col = tgt_col if tgt_col else src_col
        if tgt_type and col in df.columns:
            pd_type = _TSQL_TO_PANDAS.get(tgt_type.split('(')[0].lower())
            if pd_type:
                try:
                    df[col] = df[col].astype(pd_type)
                except (ValueError, TypeError):
                    pass  # leave as-is; DDL builder will use auto-inference

    print(f"schema overrides applied: {len(_overrides)} rules, {len(rename_map)} renames")
else:
    print("no table_id — schema overrides skipped")

In [ ]:
# Cell 4c — Apply elt_transform_config rules (config-driven per-table transforms)
# Rules are fetched once per table_id and applied in sequence_number order.
# Supported transform_type values:
#   add_literal_column  target_column = rule_value (constant string)
#   remove_column       drop comma-sep columns in rule_value
#   filter_rows         pandas query string in rule_value
#   dedup               drop duplicates; rule_value = comma-sep subset cols or NULL for all cols
#   split_datetime      split source_column → date + time cols named in rule_value "date_col,time_col"
#   union_table         concat secondary bronze table whose table_id is in rule_value
#   unpivot             melt wide→long; rule_value = "id_vars=…;value_vars=…;var_name=…;val_name=…"

if table_id:
    _tx_conn = _connect_warehouse()
    try:
        with _tx_conn.cursor() as _cur:
            _cur.execute(
                "SELECT transform_type, source_column, target_column, rule_value "
                "FROM mdf_platform_orchestration.elt_transform_config "
                "WHERE table_id = ? ORDER BY sequence_number",
                (int(table_id),)
            )
            _transforms = _cur.fetchall()
    finally:
        _tx_conn.close()

    for tx_type, src_col, tgt_col, rule_val in _transforms:

        if tx_type == 'add_literal_column':
            df[tgt_col] = rule_val

        elif tx_type == 'remove_column':
            cols = [c.strip() for c in rule_val.split(',') if c.strip() in df.columns]
            df = df.drop(columns=cols)

        elif tx_type == 'filter_rows':
            df = df.query(rule_val).reset_index(drop=True)

        elif tx_type == 'dedup':
            subset = [c.strip() for c in rule_val.split(',') if c.strip()] if rule_val else None
            df = df.drop_duplicates(subset=subset).reset_index(drop=True)

        elif tx_type == 'split_datetime':
            # rule_value: "date_col,time_col"
            date_col, time_col = [c.strip() for c in rule_val.split(',')]
            df[src_col] = pd.to_datetime(df[src_col], errors='coerce')
            df[date_col] = df[src_col].dt.date
            df[time_col] = df[src_col].dt.time

        elif tx_type == 'union_table':
            # rule_value: table_id of the secondary bronze table to union in
            _ref_conn = _connect_warehouse()
            try:
                with _ref_conn.cursor() as _cur:
                    _cur.execute(
                        "SELECT datasubject, classification, sourcesystem, sourceschema, "
                        "tablename, ingest_channel, ingest_partition, bronze_file_type "
                        "FROM mdf_platform_orchestration.elt_table_config WHERE table_id = ?",
                        (int(rule_val),)
                    )
                    _ref = _cur.fetchone()
            finally:
                _ref_conn.close()
            if _ref:
                _ref_path = (f"/lakehouse/default/Files/bronze/{_ref[0]}/{_ref[1]}/"
                             f"{_ref[2]}/{_ref[3]}/{_ref[4]}/{_ref[5]}/{_ref[6]}")
                df_sec = read_bronze(_ref_path, _ref[7] or 'parquet')
                df = pd.concat([df, df_sec], ignore_index=True)

        elif tx_type == 'unpivot':
            # rule_value: "id_vars=col1,col2;value_vars=col3,col4;var_name=metric;val_name=value"
            params   = dict(p.split('=', 1) for p in rule_val.split(';'))
            id_vars  = [c.strip() for c in params.get('id_vars', '').split(',') if c.strip()]
            val_vars = [c.strip() for c in params.get('value_vars', '').split(',') if c.strip()] or None
            df = df.melt(id_vars=id_vars, value_vars=val_vars,
                         var_name=params.get('var_name', 'variable'),
                         value_name=params.get('val_name', 'value'))

    print(f"transform rules applied: {len(_transforms)}  rows after: {len(df):,}")
else:
    print("no table_id — transform rules skipped")

In [ ]:
# Cell 5 — Add audit columns + row_hash for SCD2 change detection
audit_cols = {
    'pipeline_id':           pipeline_id,
    'pipeline_name':         pipeline_name,
    'pipeline_run_id':       pipeline_run_id,
    'pipeline_trigger_id':   pipeline_trigger_id,
    'pipeline_trigger_time': pipeline_trigger_time,
    'pipeline_trigger_type': pipeline_trigger_type,
}
for k, v in audit_cols.items():
    df[k] = v

# row_hash is computed over business columns only (excludes audit + SCD2 cols)
business_cols = [c for c in df.columns if c not in audit_cols]

def _row_to_hash(r) -> bytes:
    payload = '\u241e'.join('' if pd.isna(v) else str(v) for v in r)
    return hashlib.sha256(payload.encode('utf-8')).digest()

df['row_hash'] = df[business_cols].apply(_row_to_hash, axis=1)

now_utc = datetime.now(timezone.utc).replace(tzinfo=None)
df['valid_from'] = now_utc
df['valid_to']   = pd.NaT
df['is_current'] = True

In [ ]:
# Cell 7 — Open warehouse connection for staging + silver operations
conn = _connect_warehouse()
print('warehouse connected')

In [ ]:
# Cell 7 — Connect to Fabric Warehouse via pyodbc + Entra access token
def _connect_warehouse() -> pyodbc.Connection:
    token = DefaultAzureCredential().get_token('https://database.windows.net/.default').token
    token_bytes = bytes(token, 'utf-8')
    exp_token = b''
    for b in token_bytes:
        exp_token += bytes({b}) + bytes(1)
    attrs = {1256: struct.pack('=i', len(exp_token)) + exp_token}  # SQL_COPT_SS_ACCESS_TOKEN
    conn_str = (
        'Driver={ODBC Driver 18 for SQL Server};'
        f'Server={warehouse_endpoint},1433;'
        f'Database={warehouse_name};'
        'Encrypt=yes;TrustServerCertificate=no;Connection Timeout=30;'
    )
    return pyodbc.connect(conn_str, attrs_before=attrs, autocommit=False)

def execute(conn, sql: str, params: tuple = ()):
    with conn.cursor() as cur:
        cur.execute(sql, params) if params else cur.execute(sql)

conn = _connect_warehouse()
print('warehouse connected')

In [ ]:
# Cell 8 — DDL: build CREATE TABLE from DataFrame dtypes
_PANDAS_TO_TSQL = {
    'int64':   'bigint',
    'Int64':   'bigint',
    'int32':   'int',
    'Int32':   'int',
    'float64': 'float',
    'float32': 'real',
    'bool':    'bit',
    'boolean': 'bit',
    'datetime64[ns]':        'datetime2',
    'datetime64[ns, UTC]':   'datetime2',
    'object':  'varchar(8000)',
    'string':  'varchar(8000)',
}

def tsql_type(pd_dtype: str, col_name: str) -> str:
    if col_name == 'row_hash':     return 'varbinary(32)'
    if col_name == 'valid_from':   return 'datetime2 NOT NULL'
    if col_name == 'valid_to':     return 'datetime2 NULL'
    if col_name == 'is_current':   return 'bit NOT NULL'
    return _PANDAS_TO_TSQL.get(str(pd_dtype), 'varchar(8000)')

col_defs = ',\n    '.join(f"[{c}] {tsql_type(df[c].dtype, c)}" for c in df.columns)

create_silver_sql = f"""
IF OBJECT_ID(N'{silver_fqn}', 'U') IS NULL
BEGIN
    CREATE SCHEMA IF NOT EXISTS [{silver_schema}];
    CREATE TABLE {silver_fqn} (
    {col_defs}
    );
END
"""
execute(conn, create_silver_sql)
conn.commit()
print(f"silver table ready: {silver_fqn}")

In [ ]:
# Cell 9 — Create staging table (CTAS via empty shape), then COPY INTO from parquet
execute(conn, f"DROP TABLE IF EXISTS {staging_fqn}")
execute(conn, f"CREATE TABLE {staging_fqn} ( {col_defs} )")

copy_into_sql = f"""
COPY INTO {staging_fqn}
FROM '{staging_abfss}'
WITH (
    FILE_TYPE = 'PARQUET'
)
"""
execute(conn, copy_into_sql)
conn.commit()
print(f"staging loaded: {staging_fqn}")

In [ ]:
# Cell 10 — Load silver: full-refresh, SCD2 merge, or append (no PK fallback)
def _insert_all_sql() -> str:
    cols = ', '.join(f"[{c}]" for c in df.columns)
    return f"INSERT INTO {silver_fqn} ({cols}) SELECT {cols} FROM {staging_fqn}"

if is_full_refresh:
    execute(conn, f"TRUNCATE TABLE {silver_fqn}")
    execute(conn, _insert_all_sql())
    mode = 'full_refresh'

elif scd2_enabled:
    pk_join = ' AND '.join(f"t.[{k}] = s.[{k}]" for k in business_keys)
    non_scd_cols = [c for c in df.columns if c not in ('valid_from', 'valid_to', 'is_current')]
    col_list     = ', '.join(f"[{c}]" for c in non_scd_cols)

    # Step 1: close rows whose business key matches but row_hash changed
    close_sql = f"""
    UPDATE t
       SET t.valid_to = s.valid_from,
           t.is_current = 0
      FROM {silver_fqn}  t
      JOIN {staging_fqn} s ON {pk_join}
     WHERE t.is_current = 1
       AND t.row_hash <> s.row_hash
    """
    execute(conn, close_sql)

    # Step 2: insert new versions for changed rows + brand-new business keys
    # After step 1, closed rows have is_current=0, so LEFT JOIN ON is_current=1
    # treats them as missing → they get re-inserted as the new version.
    insert_sql = f"""
    INSERT INTO {silver_fqn} ({col_list}, [valid_from], [valid_to], [is_current])
    SELECT {', '.join(f's.[{c}]' for c in non_scd_cols)},
           s.[valid_from], NULL, 1
      FROM {staging_fqn} s
      LEFT JOIN {silver_fqn} t
        ON {pk_join} AND t.is_current = 1
     WHERE t.[{business_keys[0]}] IS NULL
    """
    execute(conn, insert_sql)
    mode = 'scd2_merge'

else:
    # No business key + not full refresh → append (history preserved by valid_from)
    execute(conn, _insert_all_sql())
    mode = 'append'

conn.commit()
print(f"load mode: {mode}")

In [ ]:
# Cell 11 — Cleanup: drop staging table, remove staging parquet, close connection
try:
    execute(conn, f"DROP TABLE IF EXISTS {staging_fqn}")
    conn.commit()
finally:
    conn.close()

try:
    os.remove(staging_local)
except FileNotFoundError:
    pass

result = {
    'status':     'SUCCESS',
    'silver':     silver_fqn,
    'mode':       mode,
    'rows':       int(len(df)),
    'run_id':     pipeline_run_id,
    'partition':  ingest_partition,
}
print(json.dumps(result, indent=2))